### Import Libraries

In [ ]:
import warnings
import numpy as np
import pandas as pd
from google.colab import drive
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.datasets import make_classification

# Suppress specific warning categories
warnings.filterwarnings("ignore", category=ConvergenceWarning)

### Authenticate to GDrive

In [ ]:
# Authenticate to Google Drive to read in csv file
drive.mount('/content/drive')

Mounted at /content/drive


### Program Execution

In [ ]:
class MembershipInferenceAttack:

  # Default constructor
  def __init__(self, shadow_m_total, prep_train_features, prep_train_label, prep_test_features, prep_test_labels):
    # Define total shadow models to be created
    self.shadow_m_total = shadow_m_total

    # Declare model list obj
    self.shadow_model_list = []

    # Declare attack model obj
    self.attack_model = None

    # Define train target model features / labels
    self.X_train_target = prep_train_features
    self.y_train_target = prep_train_label
    self.X_test_target = prep_test_features
    self.y_test_target = prep_test_labels


  # Train the target model (f) which will be attack
  # Input params: training and test set
  def train_target_model(self, model_type, X_train, y_train, X_test, y_test):

    # Check target model to be trained
    if model_type == "LogisticRegression":
      # Define model to be trained
      self.target_model = LogisticRegression(max_iter=1000, random_state=42)

      # Train the target model using the training set
      self.target_model.fit(X_train, y_train.values.ravel())

      # Store training and test data for the target
      self.target_train = (X_train, y_train)
      self.target_test = (X_test, y_test)

      # Compute model accuracy
      accuracy = self.target_model.score(X_test, y_test)

      print(f"Target model accuracy: {accuracy}")

      # Return accuracy
      return accuracy

  # Compute the prediction loss metric for each sample (row)
  # Input params: target_model, features, labels
  def compute_prediction_loss(self, target_model, X, y):

    # Epilson value for numerical stability
    epsilon = 1e-10

    # Compute the probability for each sample (row) with respect to the feature classification
    probs = target_model.predict_proba(X)

    # Get probability of the true class for each sample
    true_feature_probs = probs[np.arange(len(y)), y]

    # Compute Cross-Entropy Loss for each sample
    # A lower loss value means a better, more confident prediction.
    loss_vals = -np.log(true_feature_probs + epsilon)

    # Return loss
    return loss_vals

  # Compute the prediction loss metric for each sample (row)
  # Input params: target_model, features, labels
  def compute_prediction_confidence(self, target_model, X, y):

    # Epilson value for numerical stability
    epsilon = 1e-10

    # Compute the probability for each sample (row) with respect to the feature classification
    probs = target_model.predict_proba(X)

    # Get total samples
    samples = len(X)

    # Get maximum probability of predicted class
    max_confidence = np.max(probs, axis=1)

    # Get probability of the true class for each sample
    true_feature_confidence = probs[np.arange(samples), y]

    sorted_probs = np.sort(probs, axis=1)
    confidence_margin = sorted_probs[:, -1] - sorted_probs[:, -2]

    # Compute Cross-Entropy of predictions for each sample
    entropy = -np.sum(probs * np.log(probs + epsilon), axis=1)

    # Combine all features
    features = np.column_stack([
        max_confidence,
        true_feature_confidence,
        confidence_margin,
        entropy
    ])

    return features


  # Train shadow model
  # Input params: D-shadow
  def train_shadow_models(self, X_shadow, y_shadow):

    # Rule of thumb: Split D shadow (same sample of data used to train_target_model) into train_set (D-in) & test_set (D-out)

    # Create list to store shadow model data
    d_train_data = []
    d_test_data = []

    # Train shadow model for the total specified in class obj instantiation
    # If shadow_m_total = 5, train 5 diferent models
    for i in range(self.shadow_m_total):

      # Split D-shadow into train/test for this shadow model
      X_train, X_test, y_train, y_test = train_test_split(X_shadow, y_shadow, test_size=0.2, random_state=42)

      # Define shadow model to be trained
      shadow_model = LogisticRegression(max_iter=1000, random_state=42)

      # Train model
      shadow_model.fit(X_train, y_train.values.ravel())

      # Add save shadow model this instance of class list for easy accessibility within other functions
      self.shadow_model_list.append(shadow_model)

      # Store the train/test splits with their associated labels
      d_train_data.append((X_train, y_train, shadow_model))
      d_test_data.append((X_test, y_test, shadow_model))

      # Display shadow model training process
      if (i + 1) % self.shadow_m_total == 0:
        print(f"Trained {i + 1}/{self.shadow_m_total} shadow models")

    # Return shadow models trained via list
    return d_train_data, d_test_data


  # Prepare dataset for which the attack model was target
  # Dataset includes prediction loss values
  def prep_attack_dataset(self, d_train_data, d_test_data):

    # Single lists for storing prediction loss value
    X_attack = []
    y_attack = []

    # For the values in the d_train_data
    for X_train, y_train, shadow_model in d_train_data:

      # Calculate the prediction loss
      # Loss values will be in a 1D vector (e.g. [0.3, 0.4] etc.)
      loss_vals = self.compute_prediction_confidence(shadow_model, X_train, y_train)

      # Flatten to ensure loss_vals is in 1D array format
      loss_vals = np.array(loss_vals).flatten()

      # Add loss values to list
      X_attack.extend(loss_vals)

      # Because the assumption is that the training set will be all members
      # We create a list of ones based on the no.of samples being computed for prediction loss
      y_attack.extend([1] * len(loss_vals))


    # For the values in the d_test_data
    for X_train, y_train, shadow_model in d_test_data:

      # Calculate the prediction loss
      # Loss values will be in a 1D vector (e.g. [0.3, 0.4] etc.)
      loss_vals = self.compute_prediction_confidence(shadow_model, X_train, y_train)

      # Flatten to ensure loss_vals is in 1D array format
      loss_vals = np.array(loss_vals).flatten()
      # Add loss values to list
      X_attack.extend(loss_vals)

      # Because the assumption is that the training set will be all members
      # We create a list of zeros based on the no.of samples being computed for prediction loss
      y_attack.extend([0] * len(loss_vals))

    X_attack = np.array(X_attack, dtype=np.float64).reshape(-1, 1)
    y_attack = np.array(y_attack, dtype=np.int32)

    # Return loss predictions & labels
    return X_attack, y_attack


  # Train the attack model
  # Input params: X_attack, y_attack
  def train_attack_model(self, X_attack, y_attack):

    # Define attack model to be trained
    self.attack_model = LogisticRegression(max_iter=1000, random_state=42)

    # Train the attack model
    self.attack_model.fit(X_attack, y_attack.ravel())

    # Compute model accuracy
    accuracy = self.attack_model.score(X_attack, y_attack)

    print(f"Attack model accuracy: {accuracy}")


  # Perform Membership Inference attack on target model
  def execute_attack(self):

    # Retrieve target model training and test set
    X_train, y_train = self.target_train
    X_test, y_test = self.target_test

    # Compute prediction loss values for target model's training data (actual members)
    train_losses = self.compute_prediction_confidence(self.target_model, X_train, y_train)

    # Compute losses for target model's test data (actual non-members)
    test_losses = self.compute_prediction_confidence(self.target_model, X_test, y_test)

    # Use attack model to predict membership
    train_predictions = self.attack_model.predict(train_losses.reshape(-1, 1))
    test_predictions = self.attack_model.predict(test_losses.reshape(-1, 1))

    # Calculate attack accuracy
    train_acc = accuracy_score([1] * len(train_predictions), train_predictions)
    test_acc = accuracy_score([0] * len(test_predictions), test_predictions)
    overall_acc = (train_acc * len(train_predictions) + test_acc * len(test_predictions)) / \
                  (len(train_predictions) + len(test_predictions))

    return {
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'overall_accuracy': overall_acc,
        'train_losses': train_losses,
        'test_losses': test_losses
    }


# Example usage and demonstration
if __name__ == "__main__":

  # Preprocessed Training Set
  prep_train_features = pd.read_csv('/content/drive/MyDrive/Masters/Preprocessed_Dataset/X_train.csv')
  prep_train_labels = pd.read_csv('/content/drive/MyDrive/Masters/Preprocessed_Dataset/y_train.csv')

  # Preprocessed Test Set
  prep_test_features = pd.read_csv('/content/drive/MyDrive/Masters/Preprocessed_Dataset/X_test.csv')
  prep_test_labels = pd.read_csv('/content/drive/MyDrive/Masters/Preprocessed_Dataset/y_test.csv')

  # Perform set splitting
  # Split target data into d-train/d-test
  X_target, X_shadow, y_target, y_shadow = train_test_split(prep_train_features, prep_train_labels, test_size=0.25, random_state=42)

  # Setup
  print("1. Setup Membership Inference Attack Params...")
  mia = MembershipInferenceAttack(1, prep_train_features, prep_train_labels, prep_test_features, prep_test_labels)

  # Train target model
  print("\n2. Training Target Model...")
  mia.train_target_model("LogisticRegression", mia.X_train_target, mia.y_train_target, mia.X_test_target, mia.y_test_target)

  # Train shadow models
  print("\n3. Training Shadow Models...")
  d_train, d_test = mia.train_shadow_models(X_shadow, y_shadow)

  # Prepare attack dataset
  print("\n4. Preparing dataset for performing attacks...")
  X_attack, y_attack = mia.prep_attack_dataset(d_train, d_test)

  #print("X shape:", X_attack.shape)
  #print("y shape:", y_attack.shape)

  # Train attack model
  print("\n5. Train Attack Model...")
  mia.train_attack_model(X_attack, y_attack)

  # Perform the attack
  print("\n6. Performing Attack...")
  results = mia.execute_attack()

  print("\n7. Get Attack Results...")
  print(f"Attack on [members] accuracy: {results['train_accuracy']}")
  print(f"Attack on [non-members] accuracy: {results['test_accuracy']}")
  print(f"Overall accuracy: {results['overall_accuracy']}")

  print("\n7. Threshold Analysis:")

  # Compare accuracy vs baseline of 0.72
  if results['overall_accuracy'] > 0.72:
    print("The MIA attack is successful! The target model is vulnerable to privacy leakage.")
  else:
    print("The MIA attack is not very effective. The target model is less vulnerable to privacy leakage.")

1. Setup Membership Inference Attack Params...

2. Training Target Model...
Target model accuracy: 0.8435931578408277

3. Training Shadow Models...
Trained 1/1 shadow models

4. Preparing dataset for performing attacks...

5. Train Attack Model...
Attack model accuracy: 0.9410843355321235

6. Performing Attack...

7. Get Attack Results...
Attack on [members] accuracy: 1.0
Attack on [non-members] accuracy: 0.0
Overall accuracy: 0.9411580405371207

7. Threshold Analysis:
The MIA attack is successful! The target model is vulnerable to privacy leakage.
